# NYC Yellow Taxi Fare Prediction using PySpark
This notebook processes large-scale NYC taxi trip data to predict the total fare amount using a **Linear Regression** model. The implementation uses **PySpark ML** for scalable, big-data processing.

### Steps Covered:
1. Initialize Spark Session
2. Load Dataset
3. Feature Engineering (`VectorAssembler`)
4. Train/Test Data Splitting
5. Model Training
6. Evaluation & Metrics
7. Saving the Model


## 1.Initialize the Spark Session
We start by building a Spark session. This serves as the entry point to interact with Spark's distributed dataframes and machine learning libraries.

In [1]:
!pip install findspark
!pip install pyspark

In [2]:
import findspark
findspark.init()

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYC_Taxi_Fare_Prediction") \
    .getOrCreate()

## 2 Load the Dataset
We load the NYC Yellow Taxi data. Since taxi data is large and structured, it is typically stored in the optimized **Parquet** format.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df = spark.read.parquet("/content/drive/MyDrive/part-00000-3f938790-288e-4ac3-923c-c14c54bcaa6f-c000.snappy.parquet")

print("Initial Data Schema:")
df.printSchema()

Initial Data Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- duration_minutes: double (nullable = true)
 |-- hour: integer (nullable = true)
 |-- day_of_

In [5]:
print(f'There are {df.count()} rows in the dataset')
print(f'There are {len(df.columns)} columns in the dataset')

There are 10803240 rows in the dataset
There are 23 columns in the dataset


## 3.Feature Engineering (Vector Assembler)
Spark ML algorithms require all input features to be combined into a single, dense vector column. We use the `VectorAssembler` to merge `trip_distance` and `passenger_count` into a new column called `features`.

In [6]:
from pyspark.ml.feature import VectorAssembler

# Spark ML requires input features to be bundled into a single vector column
assembler = VectorAssembler(
    inputCols=["trip_distance", "passenger_count"],
    outputCol="features"
)

ml_data = assembler.transform(df)

print("Data ready for Machine Learning (with 'features' vector):")
ml_data.select("features", "fare_amount").show(5)

Data ready for Machine Learning (with 'features' vector):
+----------+-----------+
|  features|fare_amount|
+----------+-----------+
|[5.52,2.0]|       19.0|
|[7.45,2.0]|       26.0|
| [1.2,1.0]|        9.0|
| [6.0,1.0]|       18.0|
|[3.21,1.0]|       11.5|
+----------+-----------+
only showing top 5 rows


## 4.Train / Test Split
We split our prepared data into a **Training Set (80%)** to teach the model and a **Testing Set (20%)** to evaluate its real-world performance. A random seed is set for reproducibility.

In [7]:
trainDF, testDF = ml_data.randomSplit([0.8, 0.2], seed = 42)

print(f'There are {trainDF.count()} rows in the training set, and {testDF.count()} in the test set')

There are 8642122 rows in the training set, and 2161118 in the test set


## 5.Train the Linear Regression Model
We initialize a `LinearRegression` model, pointing it to our `features` vector and our target `fare_amount`. We then fit the model to our training data to learn the optimal line coefficients and intercept.

In [8]:
from pyspark.ml.regression import LinearRegression

# Initialize the model specifying the features and the target label
lr = LinearRegression(featuresCol="features", labelCol="fare_amount")

# Fit the model on the training dataset
lr_model = lr.fit(trainDF)

# Print the learned coefficients and intercept
print(f"Coefficients: {lr_model.coefficients}")
print(f"Intercept: {lr_model.intercept}")

Coefficients: [2.755550903996806,-0.011827288267080153]
Intercept: 4.408078946228743


## 6.Evaluate Model Performance
We use the trained model to make predictions on the unseen test data. To determine how well the model performed, we calculate:
* **Root Mean Squared Error (RMSE):** The average prediction error in dollars.
* **R-squared ($R^2$):** The percentage of fare variance explained by our features.

In [9]:
from pyspark.ml.evaluation import RegressionEvaluator

# Make predictions on the test dataset
predictions = lr_model.transform(testDF)

print("Sample Predictions vs Actual Fares:")
predictions.select("trip_distance", "fare_amount", "prediction").show(10)

# Evaluate model performance using RMSE (Root Mean Squared Error) and R2
evaluator_rmse = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")

rmse = evaluator_rmse.evaluate(predictions)
r2 = evaluator_r2.evaluate(predictions)

print("--- Model Metrics ---")
print(f"Root Mean Squared Error (RMSE) on test data: {rmse:.4f}")
print(f"R-squared (R2) on test data: {r2:.4f}")

Sample Predictions vs Actual Fares:
+-------------+-----------+------------------+
|trip_distance|fare_amount|        prediction|
+-------------+-----------+------------------+
|          3.7|       14.0|14.591790002749844|
|          4.9|       19.0|17.886623799278933|
|          5.3|       18.0|18.977016872610573|
|          3.4|       14.5|13.765124731550802|
|         15.4|       41.5|46.831735579512475|
|          1.2|        6.5|  7.70291274275783|
|          8.7|       25.0| 28.35771723446679|
|          2.7|       13.0| 11.83623909875304|
|          0.2|        3.5| 4.947361838761024|
|          0.9|        8.5| 6.876247471558788|
+-------------+-----------+------------------+
only showing top 10 rows
--- Model Metrics ---
Root Mean Squared Error (RMSE) on test data: 3.1509
R-squared (R2) on test data: 0.9088


## 7.Save the Trained Model
To avoid retraining the model on massive amounts of data in the future, we persist the trained model directory directly to disk so it can be reloaded instantly later.

In [10]:
import json

# Extract the coefficients and intercept from your Spark model
model_data = {
    "intercept": float(lr_model.intercept),
    "coefficients": [float(c) for c in lr_model.coefficients],
    "features_order": ["trip_distance", "passenger_count"]
}

# Open a standard single file and dump the data as JSON
with open("nyc_taxi_lr_model.json", "w") as f:
    json.dump(model_data, f, indent=4)

print("Model saved successfully as a single file: 'nyc_taxi_lr_model.json'")

Model saved successfully as a single file: 'nyc_taxi_lr_model.json'
